<a href="https://colab.research.google.com/github/Varshini151205/DL/blob/main/DL_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Implementation of BERT model**

In [6]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Step 1: Load model
model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

# Step 2: Input text
text = "I love this product!"

# Step 3: Tokenize
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Step 4: Prediction
outputs = model(**inputs)
logits = outputs.logits

# Step 5: Convert to probabilities
probabilities = torch.nn.functional.softmax(logits, dim=-1)

# Step 6: Get predicted class
predicted_class = torch.argmax(probabilities, dim=-1).item()

# Step 7: Print result
if predicted_class == 1:
    print("Sentiment: Positive ")
else:
    print("Sentiment: Negative")

# Confidence
confidence = probabilities[0][predicted_class].item()
print("Confidence:", confidence)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sentiment: Positive 
Confidence: 0.6009165048599243


In [21]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

# load tokenizer and model (Masked LM)
model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)

# Input with [MASK]
text = "I love this [MASK]!"

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Find mask index
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

#  Predict
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Get predicted word
mask_logits = logits[0, mask_token_index, :]
top_token_id = torch.argmax(mask_logits, dim=1)

predicted_word = tokenizer.decode(top_token_id)

print("Original Sentence:", text)
print("Predicted Word:", predicted_word)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original Sentence: I love this [MASK]!
Predicted Word: place


In [20]:
top_k = 5

mask_logits = logits[0, mask_token_index, :]
top_tokens = torch.topk(mask_logits, top_k, dim=1).indices[0].tolist()

print("Top Predictions:")
for token in top_tokens:
    print(tokenizer.decode([token]))

Top Predictions:
place
house
man
girl
song


In [22]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F

# load tokenizer and model
model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)

# Input
text = "I love this [MASK]!"

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Find mask index
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Convert logits → probabilities
mask_logits = logits[0, mask_token_index, :]
probs = F.softmax(mask_logits, dim=-1)

# Get top prediction
top_token_id = torch.argmax(probs, dim=1)
predicted_word = tokenizer.decode(top_token_id)

# Get confidence
confidence = probs[0, top_token_id].item()

print("Original Sentence:", text)
print("Predicted Word:", predicted_word)
print("Confidence:", round(confidence * 100, 2), "%")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Original Sentence: I love this [MASK]!
Predicted Word: place
Confidence: 35.85 %


In [23]:
texts = [
    "I love this product",
    "This food is delicious",
    "I hate this service",
    "This movie is amazing"
]

In [24]:
def mask_text(text, tokenizer):
    tokens = tokenizer.tokenize(text)
    import random

    idx = random.randint(0, len(tokens)-1)
    original_word = tokens[idx]
    tokens[idx] = "[MASK]"

    return tokenizer.convert_tokens_to_string(tokens), original_word

In [25]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F
import random

# Load model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

texts = [
    "I love this product",
    "This food is delicious",
    "I hate this service",
    "This movie is amazing"
]

# Training loop
for epoch in range(5):
    total_loss = 0

    for text in texts:
        masked_text, original_word = mask_text(text, tokenizer)

        inputs = tokenizer(masked_text, return_tensors="pt")
        labels = inputs["input_ids"].clone()

        # Only compute loss on masked token
        mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
        labels[:] = -100
        labels[mask_token_index] = tokenizer.convert_tokens_to_ids(original_word)

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1, Loss: 16.5223
Epoch 2, Loss: 28.7232
Epoch 3, Loss: 12.4224
Epoch 4, Loss: 11.9764
Epoch 5, Loss: 13.1941


In [26]:
text = "I love this [MASK]!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

probs = F.softmax(logits[0, mask_token_index, :], dim=-1)

top_token_id = torch.argmax(probs, dim=1)
predicted_word = tokenizer.decode(top_token_id)

confidence = probs[0, top_token_id].item()

print("Predicted:", predicted_word)
print("Confidence:", confidence)

Predicted: service
Confidence: 0.15065805613994598


**Implementation of transformer model**

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
#Sample Text Data
texts = [
    "i love this product",
    "i hate movies",
    "i like eating"
]
labels = torch.tensor([1,0, 1])  # 1 = Positive, 0 = Negative
# Build Vocabulary

words = []
for text in texts:
    words.extend(text.split())

vocab = {word: idx+1 for idx, word in enumerate(set(words))}
vocab["<PAD>"] = 0

# 3. Encode Text
def encode(text, vocab, max_len=10):
    tokens = text.split()
    seq = [vocab.get(word, 0) for word in tokens]

    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    return seq

encoded_data = [encode(text, vocab) for text in texts]
input_data = torch.tensor(encoded_data)

# 4. Positional Encoding

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# =========================
# 5. Transformer Model
# =========================
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, num_classes):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=256,
            dropout=0.1
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)

        x = x.permute(1, 0, 2)
        x = self.transformer(x)

        x = x.mean(dim=0)
        x = self.dropout(x)
        x = self.fc(x)

        return x


# Initialize Mode
vocab_size = len(vocab)
model = TransformerClassifier(
    vocab_size=vocab_size,
    d_model=128,
    n_heads=4,
    num_layers=2,
    num_classes=2
)
# Training Setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 8. Training Loop

for epoch in range(50):
    optimizer.zero_grad()

    outputs = model(input_data)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

# 9. Prediction
outputs = model(input_data)
probs = F.softmax(outputs, dim=-1)
predicted = torch.argmax(probs, dim=1)

# 10. Display Results

for i, text in enumerate(texts):
    label = "Positive " if predicted[i] == 1 else "Negative "
    confidence = probs[i][predicted[i]].item()

    print(f"Text: {text}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.4f}")
    print("----------------------")

/tmp/ipykernel_3590/217759059.py:73: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer = nn.TransformerEncoder(


Epoch 0, Loss: 0.7202698588371277
Epoch 10, Loss: 0.03137289360165596
Epoch 20, Loss: 0.0007201577536761761
Epoch 30, Loss: 0.0005844164406880736
Epoch 40, Loss: 0.0004539187066257
Text: i love this product
Prediction: Positive 
Confidence: 0.9995
----------------------
Text: i hate movies
Prediction: Negative 
Confidence: 0.9991
----------------------
Text: i like eating
Prediction: Positive 
Confidence: 0.9997
----------------------


**Implement ViT model**